In [ ]:
!pip install grpcio==1.42.0 tensorflow-serving-api==2.7.0
!pip install keras-image-helper

In [8]:
import grpc

import tensorflow as tf

from tensorflow_serving.apis import predict_pb2
from tensorflow_serving.apis import prediction_service_pb2_grpc, prediction_service_pb2


In [2]:
host = 'localhost:8500'

channel = grpc.insecure_channel(host)

In [9]:
from tensorflow_serving.apis import prediction_service_pb2
stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

In [21]:
from keras_image_helper import create_preprocessor

preprocessor = create_preprocessor('xception', target_size=(150, 150))

X = preprocessor.from_path('../../data/eval/starfruit/starfruit_00141.jpg')

In [22]:
def np_to_protobuf(data):
    return tf.make_tensor_proto(data, shape=data.shape)

In [32]:
pb_request = predict_pb2.PredictRequest()

pb_request.model_spec.name = 'model'
pb_request.model_spec.signature_name = 'serving_default'

pb_request.inputs['input_2'].CopyFrom(np_to_protobuf(X))

In [29]:
pb_response = stub.Predict(pb_request, timeout=20.0)

In [35]:
preds = pb_response.outputs['dense_1'].float_val

In [36]:
classes = [
    'ambarella',
    'grapefruit',
    'jackfruit',
    'orange',
    'starfruit'
]
dict(zip(classes, preds))

{'ambarella': 1.4808247089385986,
 'grapefruit': -3.8408119678497314,
 'jackfruit': -4.034265041351318,
 'orange': -0.6728847026824951,
 'starfruit': 9.46235179901123}